# 🔤 Notebook 02 — Análise Textual e NLP

**Projeto:** Revisão Sistemática — IA na Educação  
**Protocolo:** PRISMA 2020 (`docs/protocol/prisma_protocol.md`)  
**Etapa:** 2 de 3 — Análise Textual  

---

Este notebook realiza o **processamento de linguagem natural (NLP)** sobre os títulos e abstracts dos artigos triados, identificando:

1. Entidades nomeadas (tecnologias de IA, termos educacionais)
2. Clustering temático via TF-IDF + K-Means
3. Detecção de palavras-chave de impacto, desigualdade e ética
4. Exportação da tabela enriquecida para `data/processed/papers_analisados.csv`

In [ ]:
# ===========================================================================
# CÉLULA 1 — Configuração para execução no Google Colab
# ===========================================================================
# Descomente as linhas abaixo se estiver executando no Google Colab:

# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/ia_educacao_research')
# !pip install pandas spacy scikit-learn tqdm
# !python -m spacy download en_core_web_sm

print('✅ Configuração do ambiente concluída.')

In [ ]:
# ===========================================================================
# CÉLULA 2 — Importações
# ===========================================================================
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    SPACY_AVAILABLE = True
    print(f'✅ spaCy {spacy.__version__} carregado.')
except Exception as e:
    SPACY_AVAILABLE = False
    print(f'⚠️  spaCy não disponível: {e}')
    print('   Execute: python -m spacy download en_core_web_sm')

In [ ]:
# ===========================================================================
# CÉLULA 3 — Carregamento dos Dados Triados
# ===========================================================================
INPUT_PATH = Path('../data/processed/papers_triados.csv')

if INPUT_PATH.exists():
    df = pd.read_csv(INPUT_PATH, encoding='utf-8')
    print(f'✅ Dados carregados: {len(df)} artigos triados')
else:
    print('⚠️  Execute primeiro o notebook 01_coleta_dados.ipynb')
    # Dados de exemplo
    df = pd.DataFrame({
        'paperId': ['abc123', 'def456', 'ghi789', 'jkl012'],
        'title': [
            'AI Tutoring Systems in Higher Education',
            'Equity and Algorithmic Bias in Online Learning',
            'ChatGPT for Language Learning: An Empirical Study',
            'Machine Learning for Automated Essay Scoring'
        ],
        'abstract': [
            'This study examines intelligent tutoring systems impact on student performance in university settings.',
            'We analyze algorithmic bias in adaptive learning platforms and its effects on underrepresented students.',
            'An empirical study of large language models for language acquisition among non-native speakers.',
            'Automated essay scoring using transformer models shows high correlation with human raters.'
        ],
        'year': [2022, 2023, 2024, 2021],
        'venue': ['Computers & Education', 'AI & Society', 'Language Learning', 'AIED'],
        'citationCount': [45, 89, 12, 67]
    })
    print(f'📋 Usando dados de exemplo: {len(df)} artigos')

In [ ]:
# ===========================================================================
# CÉLULA 4 — Detecção de Palavras-Chave Temáticas
# (Impacto, Desigualdade, Ética — conforme protocolo PRISMA)
# ===========================================================================

# Dicionário de termos de interesse sociológico
KEYWORDS = {
    'impacto':       ['impact', 'outcome', 'performance', 'achievement', 'improvement'],
    'desigualdade':  ['equity', 'bias', 'inequality', 'underrepresented', 'access', 'digital divide'],
    'etica':         ['ethics', 'ethical', 'fairness', 'transparency', 'accountability', 'privacy'],
    'ia_generativa': ['chatgpt', 'gpt', 'llm', 'large language model', 'generative ai', 'bard', 'gemini'],
    'tutor_ia':      ['intelligent tutoring', 'adaptive learning', 'personalized learning', 'recommendation system']
}

def detect_keywords(text: str, keywords: list) -> bool:
    """Verifica se algum keyword aparece no texto."""
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    return any(kw in text_lower for kw in keywords)

df['text_combined'] = (df['title'].fillna('') + ' ' + df['abstract'].fillna('')).str.lower()

for category, keywords in KEYWORDS.items():
    df[f'flag_{category}'] = df['text_combined'].apply(lambda t: detect_keywords(t, keywords))

print('Flags criadas:')
for category in KEYWORDS:
    count = df[f'flag_{category}'].sum()
    print(f'  flag_{category}: {count} artigos ({count/len(df)*100:.1f}%)')

In [ ]:
# ===========================================================================
# CÉLULA 5 — Clustering Semântico (TF-IDF + K-Means)
# ===========================================================================
N_CLUSTERS = 5  # Ajustar conforme o corpus

# Vetorização TF-IDF
vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words='english',
    ngram_range=(1, 2)
)
tfidf_matrix = vectorizer.fit_transform(df['text_combined'])

# K-Means
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(tfidf_matrix)

# Termos mais relevantes por cluster
feature_names = vectorizer.get_feature_names_out()
print('Termos mais relevantes por cluster:')
for i in range(N_CLUSTERS):
    cluster_center = kmeans.cluster_centers_[i]
    top_indices = cluster_center.argsort()[-10:][::-1]
    top_terms = [feature_names[j] for j in top_indices]
    print(f'  Cluster {i}: {', '.join(top_terms)}')

In [ ]:
# ===========================================================================
# CÉLULA 6 — Exportação
# ===========================================================================
OUTPUT_PATH = Path('../data/processed/papers_analisados.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

cols_to_drop = ['text_combined']
df.drop(columns=[c for c in cols_to_drop if c in df.columns]).to_csv(
    OUTPUT_PATH, index=False, encoding='utf-8'
)

print(f'✅ Dados exportados para: {OUTPUT_PATH}')
print(f'   Total de artigos analisados: {len(df)}')
print(f'   Próximo passo: Execute o notebook 03_visualizacao_resultados.ipynb')